# Notebook 03 — System rankings (RQ4)

Scores every judged query–passage pair (about 6,000 pairs in six languages) with **13 retrieval systems**: BM25 (word and character 4-gram), six multilingual dense retrievers, two cross-encoder re-rankers, two rank fusions, and a random baseline. Because every passage in each query's pool has both human and LLM labels, each system can be evaluated with *complete* judgments under both label sets — so we can test whether LLM labels (before and after calibration) rank the systems the same way humans do.

**Kaggle setup:** GPU T4 x2 (or P100), Internet On. No `HF_TOKEN` needed (all models are public).

**Inputs:** click **Add Input → Your Work → notebook 02** and attach **two versions**: one control-language version (it contains `pairs_en/sw/ar.jsonl`) and the Llama rerun version (it contains `pairs_hi/bn/te.jsonl`). Cell 2 prints which files it found.

**Output:** `results/system_scores.csv`. The final cell prints nDCG@5 with human labels as a sanity check. Send me `system_scores.csv`; I'll run the Kendall τ / τ_AP comparison against all your judge labels.

## 0. Install packages

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas<3", "scikit-learn<1.9",
                "-U", "sentence-transformers", "transformers", "accelerate"], check=True)

import os, gc, glob, json, math, re, string, time
from collections import Counter, defaultdict
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np
import pandas as pd
import torch

## 1. Configuration

In [ ]:
CONFIG = {
    "languages": ["hi", "bn", "te", "en", "sw", "ar"],
    "output_dir": "/kaggle/working/results" if os.path.exists("/kaggle") else "results",
    "batch_size": 32,
    "random_seed": 42,
    # Systems to run. Remove any that fail and re-run: results are saved per system.
    "dense_models": {
        "e5-small":  ("intfloat/multilingual-e5-small", "query: ", "passage: "),
        "e5-base":   ("intfloat/multilingual-e5-base", "query: ", "passage: "),
        "e5-large":  ("intfloat/multilingual-e5-large", "query: ", "passage: "),
        "bge-m3":    ("BAAI/bge-m3", "", ""),
        "labse":     ("sentence-transformers/LaBSE", "", ""),
        "mpnet-ml":  ("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", "", ""),
    },
    "cross_encoders": {
        "mminilm-ce":   "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
        "bge-reranker": "BAAI/bge-reranker-v2-m3",
    },
    "fusions": {                       # reciprocal-rank fusion of two systems
        "rrf-bm25-e5large": ("bm25-word", "e5-large"),
        "rrf-bm25-bgem3":   ("bm25-word", "bge-m3"),
    },
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU")

## 2. Load judged pairs (attach notebook 02 outputs via Add Input)

In [ ]:
PAIRS = {}
for lang in CONFIG["languages"]:
    found = sorted(glob.glob(f"/kaggle/input/**/pairs_{lang}.jsonl", recursive=True)) + \
            sorted(glob.glob(f"**/pairs_{lang}.jsonl", recursive=True))
    if not found:
        print(f"WARNING: pairs_{lang}.jsonl not found - attach the notebook 02 version that produced it")
        continue
    with open(found[0], encoding="utf-8") as f:
        PAIRS[lang] = [json.loads(l) for l in f]
    n_q = len({p["qid"] for p in PAIRS[lang]})
    print(f"{lang}: {len(PAIRS[lang])} pairs, {n_q} queries  <- {found[0]}")


def doc_text(p):
    return (p["title"] + ". " + p["text"]) if p.get("title") else p["text"]

## 3. Lexical systems (BM25 over each language's pooled passages) + random

In [ ]:
PUNCT = string.punctuation + "।॥‘’“”«»،؛؟"


def words(t):
    return [w.strip(PUNCT) for w in t.lower().split() if w.strip(PUNCT)]


def char4(t):
    t = " ".join(words(t))
    return [t[i:i + 4] for i in range(max(len(t) - 3, 1))]


def bm25_system(pairs, tok, k1=0.9, b=0.4):
    docs = {}
    for p in pairs:
        docs.setdefault(p["docid"], tok(doc_text(p)))
    N = len(docs)
    avgdl = sum(len(d) for d in docs.values()) / N
    df = Counter(t for d in docs.values() for t in set(d))
    idf = {t: math.log(1 + (N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
    tf = {d: Counter(toks) for d, toks in docs.items()}
    scores = []
    for p in pairs:
        dl, c = len(docs[p["docid"]]), tf[p["docid"]]
        s = 0.0
        for t in set(tok(p["query"])):
            if t in c:
                s += idf.get(t, 0) * c[t] * (k1 + 1) / (c[t] + k1 * (1 - b + b * dl / avgdl))
        scores.append(s)
    return scores


SCORES = defaultdict(dict)          # SCORES[lang][system] = list aligned with PAIRS[lang]
for lang, pairs in PAIRS.items():
    SCORES[lang]["bm25-word"] = bm25_system(pairs, words)
    SCORES[lang]["bm25-char4"] = bm25_system(pairs, char4)
    rng = np.random.default_rng(CONFIG["random_seed"])
    SCORES[lang]["random"] = rng.random(len(pairs)).tolist()
print("Lexical systems done")

## 4. Dense bi-encoders and cross-encoders (one model on the GPU at a time)

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

dev = "cuda" if torch.cuda.is_available() else "cpu"


def free():
    gc.collect()
    torch.cuda.empty_cache()


for name, (model_id, qpre, ppre) in CONFIG["dense_models"].items():
    t0 = time.time()
    try:
        m = SentenceTransformer(model_id, device=dev)
        m.max_seq_length = min(m.max_seq_length or 512, 512)
        if dev == "cuda":
            m.half()
        for lang, pairs in PAIRS.items():
            qs = sorted({p["query"] for p in pairs})
            ds = sorted({p["docid"]: doc_text(p) for p in pairs}.items())
            qe = m.encode([qpre + q for q in qs], batch_size=CONFIG["batch_size"],
                          normalize_embeddings=True, convert_to_numpy=True)
            de = m.encode([ppre + t for _, t in ds], batch_size=CONFIG["batch_size"],
                          normalize_embeddings=True, convert_to_numpy=True)
            qi = {q: i for i, q in enumerate(qs)}
            di = {d: i for i, (d, _) in enumerate(ds)}
            SCORES[lang][name] = [float(qe[qi[p["query"]]] @ de[di[p["docid"]]]) for p in pairs]
        print(f"{name}: done in {time.time() - t0:.0f}s")
    except Exception as e:
        print(f"{name}: FAILED ({e}) - skipping")
    finally:
        m = None
        free()

for name, model_id in CONFIG["cross_encoders"].items():
    t0 = time.time()
    try:
        ce = CrossEncoder(model_id, device=dev, max_length=512)
        if dev == "cuda":
            ce.model.half()
        for lang, pairs in PAIRS.items():
            s = ce.predict([(p["query"], doc_text(p)) for p in pairs],
                           batch_size=CONFIG["batch_size"], show_progress_bar=False)
            SCORES[lang][name] = [float(x) for x in s]
        print(f"{name}: done in {time.time() - t0:.0f}s")
    except Exception as e:
        print(f"{name}: FAILED ({e}) - skipping")
    finally:
        ce = None
        free()

## 5. Fusion systems and saving

In [ ]:
def ranks_within_query(pairs, scores):
    by_q = defaultdict(list)
    for i, p in enumerate(pairs):
        by_q[p["qid"]].append(i)
    rank = [0] * len(pairs)
    for idx in by_q.values():
        for r, i in enumerate(sorted(idx, key=lambda i: (-scores[i], pairs[i]["docid"])), 1):
            rank[i] = r
    return rank


for lang, pairs in PAIRS.items():
    for name, (a, b) in CONFIG["fusions"].items():
        if a in SCORES[lang] and b in SCORES[lang]:
            ra, rb = ranks_within_query(pairs, SCORES[lang][a]), ranks_within_query(pairs, SCORES[lang][b])
            SCORES[lang][name] = [1 / (60 + x) + 1 / (60 + y) for x, y in zip(ra, rb)]

rows = []
for lang, pairs in PAIRS.items():
    for system, sc in SCORES[lang].items():
        for p, s in zip(pairs, sc):
            rows.append((lang, p["qid"], p["docid"], p["human"], system, s))
out = pd.DataFrame(rows, columns=["lang", "qid", "docid", "human", "system", "score"])
path = os.path.join(CONFIG["output_dir"], "system_scores.csv")
out.to_csv(path, index=False)
print(f"Saved {len(out)} rows, {out.system.nunique()} systems -> {path}")

## 6. Sanity check: nDCG@5 with HUMAN labels (complete judgments)

In [ ]:
def ndcg_at_k(df, label_col, k=5):
    vals = []
    for _, g in df.groupby("qid"):
        g = g.sort_values(["score", "docid"], ascending=[False, True])
        gains = g[label_col].values
        dcg = sum(gains[i] / math.log2(i + 2) for i in range(min(k, len(gains))))
        ideal = sorted(gains, reverse=True)
        idcg = sum(ideal[i] / math.log2(i + 2) for i in range(min(k, len(ideal))))
        vals.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(vals))


table = out.groupby(["lang", "system"]).apply(lambda g: ndcg_at_k(g, "human")).unstack(0).round(3)
print("\nnDCG@5 with human labels (rows = systems, columns = languages):")
print(table.sort_values(table.columns[0], ascending=False).to_string())
print("\nExpect: random lowest, cross-encoders and large dense models highest.")